# Paired Event-Window Tests

This notebook explains:

- paired t-test
- Wilcoxon signed-rank test

These compare event/post-event windows with pre-event windows inside the same geography.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "processed").exists())
DATA = ROOT / "data" / "processed" / "us"

In [ ]:
windows = pd.read_csv(DATA / "statistics" / "county_event_window_panel.csv")
one_week = windows.loc[windows["window_size_weeks"].eq(1)].dropna(
    subset=["revenue_all_pre_mean", "revenue_all_post_mean"]
)
pre = one_week["revenue_all_pre_mean"]
post = one_week["revenue_all_post_mean"]
delta = post - pre

print(len(delta), delta.mean(), delta.median())

In [ ]:
paired_t = stats.ttest_rel(post, pre)
wilcoxon = stats.wilcoxon(post, pre, zero_method="wilcox")

pd.DataFrame([
    {"test": "Paired t-test", "statistic": paired_t.statistic, "p_value": paired_t.pvalue},
    {"test": "Wilcoxon signed-rank", "statistic": wilcoxon.statistic, "p_value": wilcoxon.pvalue},
])

## Assumptions

Paired t-test:

- observations are naturally paired
- paired differences are approximately normally distributed
- pairs are independent of other pairs

Wilcoxon signed-rank:

- observations are naturally paired
- paired differences are symmetrically distributed around the median
- less sensitive to non-normality than paired t-test

## Statistical Power Note

Power depends on the number of matched event windows and the variance of paired differences. Pairing can improve power because each geography acts as its own baseline, but overlapping event windows can weaken interpretation.

## Interpretation

Positive `post - pre` means event/post windows are higher than pre-event windows. A small p-value says the average or median paired difference is unlikely to be zero under the null.